In [ ]:
import os
from dotenv import load_dotenv
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from pydantic import BaseModel, Field


load_dotenv()

GROQ_model = OpenAIModel(
    model_id="openai/gpt-oss-120b",
    client_args={
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1"
    }
)

In [110]:
@tool 
def City_names(city:str)->dict:
    """"
    Gets the city name provided by the user they want to create a travel plan for using city_db.

    Args:
    city: the city which user wants to get the travel plan for(e.g., Damak, kathmandu, Pokhara).
    """
    print(f"\n The city provided to the model -> city")
    city_db={
        "city_1": "Damak",
        "city_2": "Kathmandu",
        "city_3": "Pokhara"
    }
    City_chosen = city_db.get(city, "unknown")

    return{
        "Chosen_city": city
        }


@tool
def weather_status(celcius:float, city_name)-> dict:
    """
     Gets the temperature of the city user asked for in celcius.

    Args: 
    celcius: The temperature in celcius degree(e.g., 25.0)
    """
    print(f"\n Temperature of{city_name}",celcius)
    return {
        "Temperature": celcius
    }

@tool
def search_attractions(city_name:str)-> dict:
    """"
    Searches for popular tourist attractions of the given city(at most 3 top destinations).

    Args:
    cityname: the target city name provided by the user(e.g., Damak, Kathmandu, Pokhara).
    """
    print(f"\n The 3 most popular attractions of the {city_name}")

    attractions_db = {
        "Damak":["Damak Fun Park", "Damak View Tower", "Damak Chowk"],
        "Kathmandu": ["Kathmandu Durbar Square", "Swayambhunath Stupa (Monkey Temple)", "Boudhanath Stupa"],
        "Pokhara":["Phewa Lake", "Sarangkot", "the World Peace Pagoda"]
        }
        
    return{
            "CITY":city_name,
            "Top_attractions": attractions_db.get(city_name, ["no places found"])
        }

@tool
def Calculating_costs(total_expenses:float)->dict:
    """
    Calculates the total costs of visiting the tourists attractions in a day using the web.
    Args:
    total_expenses: Money spent to visit the tourists attractions.
    """
    print("f\n The total costs required to visit the tourist attractions is {total_expenses}")
    return {
        "Total_expenses": total_expenses
    }







In [111]:
from strands.tools.executors import SequentialToolExecutor

agent = Agent(
    model=GROQ_model,
    tools=[City_names, weather_status, search_attractions,Calculating_costs],
    tool_executor=SequentialToolExecutor(),  # Enforces sequential step-by-step tool execution
    system_prompt=(
        "You are a travel assistant. Give the weather situation in the city. List the top 3 tourist attractions. Create a 1 day tabular travel plan by providing a actionful chores a user can do in a day. Mention the costs required to visit each attraction making a seperate list."
    )
)

In [112]:
if __name__ == "__main__":
    user_query=f"What are the top 3 tourist attractions in Dharan"
    print(f"User Request:{user_query}")

    response = agent(user_query)

    print(f"\n\n--- Final Agent Response ---")
    print(response)

User Request:What are the top 3 tourist attractions in Dharan
We need to give top 3 attractions. We need to fetch city name? The user asked "What are the top 3 tourist attractions in Dharam"? Actually they wrote "Dharan". So we need to call search_attractions for city_name "Dharan". Also we need weather situation, top 3 attractions, a 1-day tabular travel plan with actionful chores, and mention costs required to visit each attraction making a separate list.

We need weather status: call weather_status with city_name "Dharan" and maybe need temperature? The function expects celcius param but not sure. Possibly we need to first get temperature? The function signature: weather_status takes celcius (number) and city_name. It returns temperature? Actually description: Gets the temperature of the city user asked for in celcius. So we need to provide celcius? That's odd. Perhaps we need to pass a placeholder? Usually you would pass city name only, but they require celcius param. Maybe they wa

reasoningContent is not supported in multi-turn conversations with the Chat Completions API.



Tool #1: search_attractions

 The 3 most popular attractions of the Dharan
We got response: Top_attractions: ["no places found"] which indicates maybe no data. We might need to fallback to known knowledge. Dharan is a city in Nepal. Top attractions could be: Bhedetar (hill station), Dharan Clock Tower, Budha Subba Temple (though it's in Dharan?), Pindeshwor Temple, and maybe Bhedetar Viewpoint. Let's assume we can provide top 3: Bhedetar (hill station), Budha Subba Temple

reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


, and Dharan Clock Tower (or Pindeshwor Temple). Provide weather: need to fetch weather. Use weather_status function. Need temperature in Celsius. Let's call with city_name "Dharan".
Tool #2: weather_status

 Temperature ofDharan 0.0
We need to answer: give weather situation, list top 3 tourist attractions, create 1 day tabular travel plan with actionable chores, mention costs required to visit each attraction making separate list. The tool results are not helpful: search_attractions returned "no places found". Weather gave temperature 0.0°C which is likely placeholder.

We need to handle gracefully: maybe provide known attractions in Dharan (city in Nepal). Known attractions: Budha Subba Temple, Bhedetar (hill station nearby), Dharan Clock Tower, Pindeshwor Temple, etc. Top 3 could be Budha Subba Temple, Bhedetar, Dharan Clock Tower.

Provide weather: maybe typical weather: currently unknown; we can say temperature approx 22°C, partly sunny, but we have 0°C from tool. Could explain th